In [9]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
import numpy as np
import re

In [ ]:
!pip install beautifulsoup4 requests pandas numpy #first install this
#if pip is not installed then download pip from this site https://bootstrap.pypa.io/get-pip.py and then open command prompt and write python get-pip.py


In [12]:
def get_title(soup):
    try:
        title = soup.find('h1', class_='Title-pdp-title').text.strip()  # Use 'soup', not 'new_soup'
    except:
        title = "error"  # Return 'error' if something goes wrong

    return title


def get_price(soup):
    try:
        # Extract the price text
        price = soup.find("span", attrs={"class": "FirstPrice"}).text.strip()

        # Remove anything that's not a digit or comma using regex and get the numeric part
        price = re.sub(r'[^\d,]', '', price)  # Removes currency symbols and non-numeric characters

    except AttributeError:
        price = ""  # Return an empty string if no price is found

    return price

def get_availability(soup):
    try:
        # Get the availability info (may need to adjust the class or tag based on your page)
        availability = soup.find("span", class_="icon-bathrooms").find_next("div").text.strip()

        # Use regex to remove any non-numeric characters (like extra spaces or text)
        availability = re.sub(r'[^\d,]', '', availability)  # Remove everything except digits and commas

    except AttributeError:
        availability = "Not Available"  # Default value if availability info is not found

    return availability


def get_location(soup):
    try:
        # Extract location info from the correct div and class
        location =soup.find("div", class_="Card-info-location").find("div", class_="item").text.strip()

    except AttributeError:
        location = " shit "  # Default value if location info is not found

    return location

def get_bedrooms(soup):
    try:
        # Extract number of bedrooms
        bedrooms = soup.find("span", class_="icon-bedrooms").find_next("div").text.strip()
    except AttributeError:
        # Default value if not found
        bedrooms = "Not Available"
    
    return bedrooms

def get_bathroom(soup):
    try:
        bathroom = soup.find("span", class_="icon-bathrooms-v4").find_next("div").text.strip()
    except AttributeError:
        bathroom = "Not Available"
    
    return bathroom




In [14]:
if __name__ == '__main__':
    URL = "https://www.bproperty.com/rent/residential/apartments/1-2-3-4-5-bedroom/?page=5"
    HEADERS = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/132.0.0.0 Safari/537.36', 'Accept-Language': 'en-US, en;q=0.5'}
    
    # Send the initial request to get the page content
    webpage = requests.get(URL, headers=HEADERS)
    soup = BeautifulSoup(webpage.content, "html.parser")
    
    # Find all links with the specified class
    links = soup.find_all("a", class_="js-listing-link")
    
    # List to store all the links and remove duplicates
    links_list = list(set([link.get('href') for link in links]))
    
    # Data dictionary to store the scraped data
    d = {"Title": [], "Location": [], "Area_sqft": [], "Bedroom": [], "Bathroom": [], "Price": []}

    # Loop through each link and extract data
    for link in links_list:
        new_webpage = requests.get(link, headers=HEADERS)
        new_soup = BeautifulSoup(new_webpage.content, "html.parser")
        
        # Extract the title
        title = get_title(new_soup)
        
        # Skip rows with "error" in title
        if title == "error":
            continue
        
        # Append extracted data to the dictionary
        d['Title'].append(get_title(new_soup))
        d['Location'].append(get_location(new_soup))
        d['Area_sqft'].append(get_availability(new_soup))                                 
        d['Bedroom'].append(get_bedrooms(new_soup))
        d['Bathroom'].append(get_bathroom(new_soup))
        d['Price'].append(get_price(new_soup))

    # Create a DataFrame from the dictionary
    amazon_df = pd.DataFrame.from_dict(d)

    # Clean the data (replace empty strings with NaN and drop rows with missing titles)
    amazon_df.replace('', np.nan, inplace=True)
    amazon_df = amazon_df.dropna(subset=['Title'])  # Drop rows with missing titles

    # Save the data to a CSV file
    amazon_df.to_csv("amazon_data.csv", header=True, index=False)

    print("Data saved to 'amazon_data.csv'.")

Data saved to 'amazon_data.csv'.


In [15]:
amazon_df

,Title,Location,Area_sqft,Bedroom,Bathroom,Price
0,A Well-constructed 1200 Sq Ft Flat Is Ready Fo...,"Vatara, Badda",1200,2,2,"21,000"
1,3100 SQ FT apartment for rent near Gulshan 1 D...,"Gulshan 1, Gulshan",3100,4,4,"100,000"
2,For Rental Purpose 900 Sq Ft Home Is Now Up To...,"Sector 11, Uttara",900,2,2,"22,000"
3,All Set For Rental Purpose This 1300 Square Fe...,"26 No. North Halishahar Ward, Halishahar",1300,3,3,"30,000"
4,A Great 1970 Sq Ft Residence For Renting Purpo...,"Panchlaish Residential Area, 16 No. Chawk Baza...",1970,4,4,"40,000"
5,For Rental Purpose 1100 Sq Ft Flat Is Now Up T...,"Block H, Banasree",1100,3,3,"20,000"
6,Get Comfortable In A 2250 Sq Ft Flat For Rent ...,"Block D, Bashundhara R-A",2250,4,4,"45,000"
7,Offering You Well Constructed 900 Sq Ft Apartm...,"Tajmahal Road, Mohammadpur",900,2,2,"21,000"
8,Make this 1350 SQ FT home your next residing l...,"Block C, Bashundhara R-A",1350,3,3,"22,000"
9,Tastefully Designed This 2100 Sq. Ft Apartment...,"Gulshan 2, Gulshan",2100,3,3,"70,000"
